<img src="https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/logo_curso.png" width="300">

# Python para Actuarios

## Día 1, bloques 3 y 4: Manejo y calidad de datos

| Día | Horario | Duración |
|---|---|---|
| Miércoles 11 de noviembre | 14:30 a 16:00 y 16:30 a 17:30 | 150 minutos |

**Objetivos**

1. Importar ficheros con distinta codificación, separador y formato numérico.
2. Convertir textos en números y en fechas, detectando los valores que no se han podido convertir.
3. Normalizar textos y categorías, y contrastarlos con un catálogo de referencia.
4. Tratar valores ausentes y registros duplicados.
5. Calcular variables actuariales a partir de fechas: edad, antigüedad y plazo pendiente.
6. Construir un conjunto de validaciones reutilizable y cuantificar el impacto de cada incidencia.
7. Medir el efecto de la calidad de datos sobre una provisión y generar un informe de incidencias.

**Cómo trabajar con este notebook**

1. El código que debes completar está marcado con `***`. Sustitúyelo y ejecuta la celda con `Mayús + Intro`.
2. Ejecuta las celdas en orden: muchas utilizan variables definidas en celdas anteriores.
3. Si Colab reinicia el entorno, ejecuta de nuevo la preparación del entorno y las celdas anteriores (*Entorno de ejecución > Ejecutar anteriores*).

**Datos**

Fecha de cálculo: 31/12/2024. Datos sintéticos, importes en euros.

| Fichero | Contenido |
|---|---|
| `hogar_extraccion.xlsx` | Extracción del sistema de pólizas de Hogar, base del cálculo de la provisión para primas no consumidas (PPNC) |
| `vida_polizas.csv` | Cartera de Vida riesgo, base de la provisión matemática, exportada desde un Excel en español |
| `vida_siniestros.xlsx` | Fallecimientos declarados |
| `provincias.csv` | Catálogo de provincias con su código INE y comunidad autónoma |

**Contenido**

Parte I, preparar los datos (bloque 3)

1. Lectura de la extracción de Hogar
2. Codificación y separador
3. Texto a número
4. Texto a fecha
5. Textos y categorías
6. Valores ausentes y duplicados
7. Variables actuariales a partir de fechas

Parte II, calidad de datos de un cierre (bloque 4)

8. Un conjunto de validaciones reutilizable
9. Validaciones de Vida
10. Impacto de las incidencias
11. Efecto sobre la provisión
12. Informe de incidencias

## Preparación del entorno

Ejecuta esta celda antes que cualquier otra. En Google Colab descarga los datos del curso y sitúa el notebook en la carpeta correcta; en una instalación local no hace nada. Si Colab reinicia el entorno, vuelve a ejecutarla.

In [ ]:
# Preparación del entorno: en Google Colab descarga el repositorio del curso (en local no hace nada)
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/python_actuarios"):
        !git clone -q --depth 1 https://github.com/PrimeReSolutions/python_actuarios.git /content/python_actuarios
    %cd /content/python_actuarios/notebooks

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATOS = Path("../datos") if Path("../datos").exists() else Path("datos")
SALIDAS = Path("../salidas") if Path("../datos").exists() else Path("salidas")
SALIDAS.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

# Parte I. Preparar los datos

## 1. Lectura de la extracción de Hogar

### Ejercicio 1.1 Fecha de cálculo

Todas las validaciones de vigencia y edad dependen de la fecha de cálculo. Defínela una sola vez, al principio, y úsala en todo el notebook.

In [ ]:
FECHA_CALCULO = pd.Timestamp("***")
FECHA_CALCULO

### 1.2 Leer y conservar el original

La extracción se lee una vez y se guarda sin modificar en `hogar_ori`. Todo el tratamiento se hace sobre una copia, de modo que siempre se puede comparar el resultado con lo que se recibió. Es la misma disciplina que no sobrescribir nunca la hoja de datos de origen en Excel.

In [ ]:
hogar_ori = pd.read_excel(DATOS / "hogar_extraccion.xlsx", sheet_name="polizas")
hogar = hogar_ori.copy()

print(hogar.shape)
hogar.dtypes

La columna `prima_anual` debería ser numérica, pero pandas la ha leído como `object`: contiene valores de tipos distintos. La celda siguiente cuenta cuántos valores hay de cada tipo.

In [ ]:
hogar["prima_anual"].map(type).value_counts()

## 2. Codificación y separador

La **codificación** define cómo se guardan en el fichero los caracteres con tilde, la ñ o el símbolo del euro. Python usa por defecto UTF-8, pero muchos ficheros generados en Windows o por sistemas antiguos usan Windows-1252 (`cp1252`) o Latin-1. Si la codificación no coincide, la lectura falla o aparecen caracteres extraños, como `CÃ¡diz` en lugar de `Cádiz`.

In [ ]:
ruta_vida = DATOS / "vida_polizas.csv"

try:
    pd.read_csv(ruta_vida, sep=";")
except UnicodeDecodeError as error:
    print("Error de codificación:", error)

### Ejercicio 2.1 Detectar la codificación

La función prueba a leer el fichero con cada codificación candidata y devuelve la primera que funciona. Completa el tipo de error que hay que capturar y el valor que se devuelve.

In [ ]:
def detectar_codificacion(ruta, candidatas=("utf-8", "cp1252")):
    """Devuelve la primera codificación que permite leer el fichero completo."""
    for codificacion in candidatas:
        try:
            with open(ruta, encoding=codificacion) as f:
                f.read()
            return ***
        except ***:
            continue
    return None

codificacion = detectar_codificacion(ruta_vida)
codificacion

### Ejercicio 2.2 Detectar el separador

El separador más probable es el carácter que más se repite en la primera línea del fichero, que contiene los nombres de columna.

In [ ]:
with open(ruta_vida, encoding=codificacion) as f:
    primera_linea = f.readline()

print(primera_linea)
separador = max([",", ";", "|", "\t"], key=primera_linea.***)
separador

### Ejercicio 2.3 Leer la cartera de Vida

Lee el fichero con la codificación y el separador detectados. Los importes usan coma decimal y punto de miles, como en el bloque 2. Las fechas se dejan como texto por ahora.

In [ ]:
vida_ori = pd.read_csv(
    ruta_vida,
    sep=***,
    encoding=***,
    decimal=",",
    thousands=".",
)
vida = vida_ori.copy()

print(vida.shape)
vida.head()

## 3. Texto a número

En la columna `prima_anual` de Hogar conviven números y textos con formato español, como `"1.234,56"`. Ni `float("1.234,56")` ni `pd.to_numeric` entienden ese formato, y sustituir solo la coma por un punto daría `"1.234.56"`, que tampoco es un número. Hay que quitar primero el punto de miles y después cambiar la coma decimal por un punto.

`pd.to_numeric(..., errors="coerce")` convierte lo que puede y deja como nulo (`NaN`) lo que no. Es cómodo, pero puede ocultar errores: por eso se compara el número de nulos antes y después de la conversión.

### Ejercicio 3.1 Convertir la prima en número

In [ ]:
texto = hogar["prima_anual"].astype(str)
formato_es = texto.str.contains(",", na=False)

normalizado = texto.where(
    ~formato_es,
    texto.str.replace(***, ***, regex=False).str.replace(***, ***, regex=False),
)
hogar["prima_anual"] = pd.to_numeric(normalizado, errors="***")

nulos_antes = hogar_ori["prima_anual"].isna().sum()
nulos_despues = hogar["prima_anual"].***().sum()
print("Valores con formato español:", formato_es.sum())
print("Nulos antes y después:", nulos_antes, nulos_despues)
print("Conversiones fallidas:", nulos_despues - nulos_antes)

## 4. Texto a fecha

Las fechas de la cartera de Vida llegan como texto `dd/mm/aaaa`. Conviene indicar siempre el formato: sin él, pandas puede interpretar `03/04/2024` como 4 de marzo en lugar de 3 de abril. Con `errors="coerce"`, los textos vacíos o no válidos quedan como `NaT`, el nulo de las fechas.

### Ejercicio 4.1 Convertir las fechas de Vida

In [ ]:
columnas_fecha = ["fecha_nacimiento", "fecha_efecto", "fecha_vencimiento"]

for columna in columnas_fecha:
    vida[columna] = pd.to_datetime(vida[columna], format="***", errors="***")

print(vida[columnas_fecha].dtypes)
vida[columnas_fecha].isna().sum()

## 5. Textos y categorías

Una misma categoría escrita de formas distintas (`"Hogar Plus"`, `"HOGAR PLUS"`, `"Hogar Plus "`) se trata como tres categorías diferentes al agrupar, igual que en una tabla dinámica de Excel. Los métodos de texto de pandas se aplican a toda la columna a través de `.str`:

| Método | Efecto |
|---|---|
| `.str.strip()` | Quita los espacios al principio y al final |
| `.str.upper()`, `.str.lower()`, `.str.title()` | Mayúsculas, minúsculas o inicial en mayúscula |
| `.str.replace(a, b)` | Sustituye un texto por otro |
| `.str.contains(a)` | Indica si contiene un texto |

In [ ]:
hogar["producto"].value_counts()

### Ejercicio 5.1 Normalizar producto y canal

Quita los espacios sobrantes y deja cada palabra con la inicial en mayúscula.

In [ ]:
hogar["producto"] = hogar["producto"].str.***().str.***()
hogar["canal"] = hogar["canal"].str.***().str.***()

print(hogar["producto"].value_counts(), "\n")
print(hogar["canal"].value_counts())

### Ejercicio 5.2 Unificar la codificación del sexo

En la cartera de Vida el sexo aparece como `H`, `M`, `Hombre`, `Mujer`, en minúsculas y con espacios. Unifícalo a `H` y `M`.

In [ ]:
print(vida["sexo"].value_counts(), "\n")

vida["sexo"] = (
    vida["sexo"]
    .str.strip()
    .str.***()
    .replace({"***": "H", "***": "M"})
)
vida["sexo"].value_counts()

### Ejercicio 5.3 Contrastar con un catálogo

Una forma segura de validar una categoría es compararla con un catálogo oficial. Lista las provincias de Hogar que no están en el catálogo del INE.

In [ ]:
provincias = pd.read_csv(DATOS / "provincias.csv", dtype={"codigo_ine": str})

fuera_catalogo = ~hogar["provincia"].isin(provincias["***"])
hogar.loc[fuera_catalogo, "provincia"].value_counts()

### Ejercicio 5.4 Corregir con una tabla de equivalencias

Corrige las variantes con un diccionario de equivalencias y comprueba que ya no queda ninguna provincia fuera del catálogo. Después añade la comunidad autónoma con un cruce validado.

In [ ]:
equivalencias = {
    "Malaga": "Málaga",
    "Cadiz": "Cádiz",
    "La Coruña": "A Coruña",
    "Vizcaya": "Bizkaia",
    "Baleares": "Illes Balears",
    "MADRID": "***",
}
hogar["provincia"] = hogar["provincia"].replace(***)

print("Fuera del catálogo:", (~hogar["provincia"].isin(provincias["provincia"])).sum())

hogar = hogar.merge(provincias[["provincia", "comunidad"]], on="provincia", how="left", validate="***")
hogar["comunidad"].value_counts().head()

## 6. Valores ausentes y duplicados

### 6.1 Valores ausentes

`isna().sum()` cuenta los nulos de cada columna. En una base de cálculo de provisiones, una prima, un capital o una fecha ausentes **no se imputan**: no hay forma razonable de estimarlos y cualquier valor inventado se trasladaría al balance. Se separan como incidencia y se comunican al área responsable de los datos.

In [ ]:
print("Hogar:")
print(hogar.isna().sum(), "\n")
print("Vida:")
print(vida.isna().sum())

### Ejercicio 6.1 Pólizas de Vida con algún dato clave ausente

`any(axis=1)` indica, para cada fila, si alguna de las columnas seleccionadas es nula.

In [ ]:
columnas_clave = ["fecha_nacimiento", "fecha_efecto", "fecha_vencimiento", "capital_asegurado"]

con_ausentes = vida[columnas_clave].isna().***(axis=1)
print("Pólizas con algún dato clave ausente:", con_ausentes.sum())

### 6.2 Duplicados

`duplicated()` marca como `True` las filas repetidas. El argumento `keep` decide cuál de las repeticiones se considera la original:

| `keep` | Qué marca como duplicado |
|---|---|
| `"first"` (por defecto) | Todas las repeticiones salvo la primera |
| `"last"` | Todas las repeticiones salvo la última |
| `False` | Todas las filas que tienen alguna repetición |

### Ejercicio 6.2 El argumento `keep`

Completa `keep` en cada columna para que el resultado coincida con su nombre.

In [ ]:
ejemplo = pd.DataFrame({"poliza": ["P1", "P2", "P2", "P3", "P3", "P3"], "prima": [100, 200, 200, 300, 310, 300]})

ejemplo["dup_first"] = ejemplo["poliza"].duplicated(keep="***")
ejemplo["dup_last"] = ejemplo["poliza"].duplicated(keep="***")
ejemplo["dup_todas"] = ejemplo["poliza"].duplicated(keep=***)
ejemplo

### Ejercicio 6.3 Duplicados en la extracción de Hogar

Compara los duplicados exactos (todas las columnas iguales) con los duplicados por número de póliza. La diferencia son pólizas repetidas con algún dato distinto, por ejemplo la prima, típicas de un reproceso: no basta con borrar una de las dos, hay que averiguar cuál es la correcta.

In [ ]:
dup_exactos = hogar.***().sum()
dup_poliza = hogar.duplicated(subset="***").sum()
print("Duplicados exactos:    ", dup_exactos)
print("Duplicados por póliza: ", dup_poliza)

repetidas = hogar["poliza"].duplicated(keep=False)
diferentes = hogar[repetidas].drop_duplicates(keep=False)
diferentes.sort_values("poliza").head(6)

## 7. Variables actuariales a partir de fechas

La resta de dos fechas da una duración (`Timedelta`). `.dt.days` la convierte en días y, dividiendo entre 365,25, se obtienen años con decimales. A partir de ahí:

1. **Edad cumplida**: parte entera de los años transcurridos desde el nacimiento.
2. **Edad actuarial**: edad al cumpleaños más próximo, es decir, los años redondeados.
3. **Antigüedad** y **plazo pendiente**: años transcurridos desde el efecto y años que faltan hasta el vencimiento.

### Ejercicio 7.1 Edad al efecto

In [ ]:
anios_al_efecto = (vida["fecha_efecto"] - vida["***"]).dt.days / 365.25

vida["edad_efecto"] = np.floor(anios_al_efecto)
vida["edad_actuarial_efecto"] = np.***(anios_al_efecto)

vida[["fecha_nacimiento", "fecha_efecto", "edad_efecto", "edad_actuarial_efecto"]].head()

### Ejercicio 7.2 Edad, antigüedad y plazo pendiente a la fecha de cálculo

In [ ]:
vida["edad_calculo"] = np.floor((FECHA_CALCULO - vida["fecha_nacimiento"]).dt.days / 365.25)
vida["antiguedad"] = (FECHA_CALCULO - vida["***"]).dt.days / 365.25
vida["plazo_pendiente"] = (vida["***"] - FECHA_CALCULO).dt.days / 365.25

vida[["edad_calculo", "antiguedad", "plazo_pendiente"]].describe()

Los valores extremos de `describe()` ya avisan de problemas: edades mínimas negativas o máximas por encima de 120 años, y plazos pendientes negativos. Son las incidencias que se tratan en la parte II.

# Parte II. Calidad de datos de un cierre

## 8. Un conjunto de validaciones reutilizable

Cada validación se expresa como una **regla**: una condición que vale `True` en las filas con incidencia. Las reglas se reúnen en un diccionario, en orden de prioridad, y una única función las aplica a cualquier base. Así las mismas reglas se reutilizan en cada cierre y en cada cartera, sin copiar celdas.

La función devuelve dos tablas:

1. **La base limpia**: las filas que no incumplen ninguna regla.
2. **El registro de incidencias**: las filas que incumplen alguna regla, con el nombre de la primera que incumplen. Asignar una sola incidencia por fila evita contar dos veces el mismo importe al medir el impacto.

In [ ]:
def aplicar_reglas(df, reglas):
    """Aplica un diccionario {nombre: máscara} y devuelve (base_limpia, incidencias)."""
    marcas = pd.DataFrame(reglas, index=df.index).fillna(False).astype(bool)
    con_incidencia = marcas.any(axis=1)
    incidencias = df.loc[con_incidencia].assign(
        incidencia=marcas.loc[con_incidencia].idxmax(axis=1)    # primera regla incumplida
    )
    return df.loc[~con_incidencia], incidencias

### Ejercicio 8.1 Reglas de la base de Hogar

Completa las reglas. Una póliza está **vigente** a la fecha de cálculo si `fecha_efecto <= FECHA_CALCULO < fecha_vencimiento`. La **tasa de prima** por mil es `prima_anual / suma_asegurada * 1000`; en estos productos está entre 0,5 y 2,5 por mil, de modo que una tasa por debajo de 0,1 o por encima de 10 delata un error en la suma asegurada.

In [ ]:
vigente = (hogar["fecha_efecto"] <= FECHA_CALCULO) & (FECHA_CALCULO < hogar["***"])
tasa = hogar["prima_anual"] / hogar["suma_asegurada"] * 1000

reglas_hogar = {
    "No vigente": ***,
    "Duplicado": hogar.duplicated(subset="***", keep="first"),
    "Prima vacía": hogar["prima_anual"].***(),
    "Prima no positiva": hogar["prima_anual"] <= ***,
    "Tasa anómala": (tasa < ***) | (tasa > ***),
}

hogar_limpia, inc_hogar = aplicar_reglas(hogar, reglas_hogar)

print("Base original:", len(hogar), "| limpia:", len(hogar_limpia), "| con incidencia:", len(inc_hogar))
inc_hogar["incidencia"].value_counts()

In [ ]:
# Tasas anómalas: sumas aseguradas con tres ceros de más o de menos
inc_hogar.loc[inc_hogar["incidencia"] == "Tasa anómala", ["poliza", "producto", "prima_anual", "suma_asegurada"]]

## 9. Validaciones de Vida

La cartera de Vida añade las validaciones propias de las fechas de nacimiento, de las que dependen la edad y, con ella, la probabilidad de fallecimiento:

1. **Fechas vacías**: sin fecha no hay edad ni duración.
2. **Fechas centinela**: valores como `01/01/1900` que algunos sistemas usan cuando el campo no se rellenó. Se considera centinela cualquier nacimiento anterior a 1910.
3. **Nacimiento posterior al efecto**: suele indicar fechas cruzadas en la captura.
4. **Menor de edad al efecto**: estos productos exigen 18 años cumplidos.
5. **Fallecidos**: una póliza con fallecimiento declarado ya no genera provisión matemática; su obligación está en la provisión de prestaciones.

### Ejercicio 9.1 Leer los fallecimientos

In [ ]:
siniestros_vida = pd.read_excel(DATOS / "***")
print(siniestros_vida.shape)
siniestros_vida.head()

### Ejercicio 9.2 Reglas de la cartera de Vida

Completa las reglas y aplica la misma función `aplicar_reglas`.

In [ ]:
vigente_vida = (vida["fecha_efecto"] <= FECHA_CALCULO) & (FECHA_CALCULO < vida["fecha_vencimiento"])

reglas_vida = {
    "Fecha vacía": vida[columnas_fecha].isna().any(axis=1),
    "Fecha centinela": vida["fecha_nacimiento"] < pd.Timestamp("***"),
    "Nacimiento posterior al efecto": vida["fecha_nacimiento"] >= vida["***"],
    "Menor de edad al efecto": vida["edad_efecto"] < ***,
    "No vigente": ~vigente_vida,
    "Duplicado": vida.duplicated(subset="poliza", keep="first"),
    "Capital vacío": vida["capital_asegurado"].isna(),
    "Capital no positivo": vida["capital_asegurado"] <= 0,
    "Fallecido": vida["poliza"].isin(siniestros_vida["***"]),
}

vida_limpia, inc_vida = ***(vida, reglas_vida)

print("Base original:", len(vida), "| limpia:", len(vida_limpia), "| con incidencia:", len(inc_vida))
inc_vida["incidencia"].value_counts()

Con la base depurada, las edades y los plazos ya son coherentes:

In [ ]:
vida_limpia[["edad_efecto", "edad_calculo", "antiguedad", "plazo_pendiente"]].describe()

## 10. Impacto de las incidencias

No todas las incidencias pesan igual: 30 duplicados en una cartera de 20 000 pólizas pueden ser irrelevantes o no, según su importe. Por eso cada incidencia se pondera por el importe afectado, en relación con el total de la base limpia: la prima en Hogar y el capital asegurado en Vida. Se toma el valor absoluto para que los importes negativos también cuenten como afectados.

El impacto se clasifica con umbrales de gestión, que cada entidad fija en su política de calidad de datos. En este ejercicio:

| Impacto sobre el total | Clasificación | Tratamiento |
|---|---|---|
| Hasta el 0,5 % | Leve | Se documenta y se corrige en el siguiente ciclo |
| Del 0,5 % al 2 % | Moderado | Se corrige antes del cierre |
| Más del 2 % | Grave | Bloquea el cierre hasta resolver la causa |

`pd.cut` asigna cada valor a un tramo, como una búsqueda aproximada (`BUSCARV` con coincidencia aproximada) sobre una tabla de umbrales.

### Ejercicio 10.1 Función de impacto

In [ ]:
def resumen_impacto(incidencias, limpia, columna):
    """Número de pólizas, importe afectado e impacto por tipo de incidencia."""
    total = limpia[columna].abs().sum()
    resumen = (
        incidencias.assign(importe=incidencias[columna].abs())
        .groupby("***")
        .agg(polizas=("poliza", "***"), importe=("importe", "***"))
        .sort_values("importe", ascending=False)
    )
    resumen["impacto_pct"] = resumen["importe"] / *** * 100
    resumen["clasificacion"] = pd.cut(
        resumen["impacto_pct"],
        bins=[-np.inf, ***, ***, np.inf],
        labels=["Leve", "Moderado", "Grave"],
    )
    return resumen

impacto_hogar = resumen_impacto(inc_hogar, hogar_limpia, "prima_anual")
impacto_hogar

### Ejercicio 10.2 Impacto en Vida

Aplica la misma función a la cartera de Vida, ponderando por el capital asegurado.

In [ ]:
impacto_vida = resumen_impacto(***, ***, "***")
impacto_vida

Las incidencias con importe 0 y varias pólizas, como las primas o capitales vacíos, no tienen impacto medible precisamente porque falta el dato. Deben revisarse aunque la tabla las clasifique como leves.

## 11. Efecto sobre la provisión

Para cerrar el círculo, se calcula la provisión para primas no consumidas (PPNC) con el método de prorrata temporal: la parte de la prima anual que corresponde al periodo de cobertura posterior a la fecha de cálculo.

$$PPNC = \text{prima anual} \times \frac{\text{fecha de vencimiento} - \text{fecha de cálculo}}{\text{fecha de vencimiento} - \text{fecha de efecto}}$$

La fracción se limita al intervalo entre 0 y 1 con `clip`, como haría cualquier cálculo que no valida los datos. Se compara el resultado sobre la base depurada con el que se obtendría sobre la extracción sin depurar.

### Ejercicio 11.1 PPNC con y sin depuración

In [ ]:
def ppnc(df):
    """PPNC por prorrata temporal a la fecha de cálculo."""
    no_consumido = (df["fecha_vencimiento"] - FECHA_CALCULO).dt.days
    duracion = (df["fecha_vencimiento"] - df["fecha_efecto"]).dt.days
    fraccion = (no_consumido / duracion).clip(***, ***)
    return (df["prima_anual"] * fraccion).sum()

ppnc_limpia = ppnc(***)
ppnc_sin_depurar = ppnc(***)

print(f"PPNC base depurada:   {ppnc_limpia:,.2f}")
print(f"PPNC sin depurar:     {ppnc_sin_depurar:,.2f}")
print(f"Diferencia:           {ppnc_sin_depurar - ppnc_limpia:,.2f}")

La diferencia procede sobre todo de las pólizas con efecto posterior a la fecha de cálculo, cuya prima entera entra como no consumida, y de los duplicados, que la cuentan dos veces; los extornos la reducen. Un error de datos que no se detecta acaba directamente en el balance.

## 12. Informe de incidencias

### Ejercicio 12.1 Generar el informe

Genera un libro con el resumen de impacto y el detalle de incidencias de cada cartera, y las dos bases depuradas. Las bases depuradas se guardan con las columnas originales, sin las calculadas en este notebook.

In [ ]:
ruta_informe = SALIDAS / "informe_calidad_datos.xlsx"

with pd.ExcelWriter(ruta_informe) as writer:
    impacto_hogar.to_excel(writer, sheet_name="Impacto Hogar")
    impacto_vida.to_excel(writer, sheet_name="Impacto Vida")
    ***.to_excel(writer, sheet_name="Incidencias Hogar", index=False)
    ***.to_excel(writer, sheet_name="Incidencias Vida", index=False)
    hogar_limpia[hogar_ori.columns].to_excel(writer, sheet_name="Hogar depurada", index=False)
    vida_limpia[***].to_excel(writer, sheet_name="Vida depurada", index=False)

pd.ExcelFile(ruta_informe).sheet_names

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(ruta_informe))

## Resumen

1. Antes de leer un fichero hay que conocer su codificación, su separador y su formato numérico; los tres pueden detectarse con unas pocas líneas de código.
2. Las conversiones de texto a número y a fecha se hacen con `errors="coerce"` y se controlan comparando los nulos antes y después.
3. Los textos se normalizan con los métodos `.str` y las categorías se contrastan con un catálogo de referencia.
4. Los datos ausentes en importes y fechas no se imputan: se separan como incidencia.
5. Las validaciones se escriben como reglas en un diccionario y una sola función las aplica a cualquier cartera.
6. El impacto de cada incidencia se mide sobre el importe afectado y se clasifica con umbrales de gestión.
7. La calidad de los datos tiene un efecto directo y cuantificable sobre las provisiones.

El día 2 empieza con la calibración de distribuciones, ya sobre datos depurados.